# Heretic on Qwen3-4B in Colab

This notebook clones the Rowan Dauria Heretic fork, installs it into the Colab runtime, writes a small `config.toml`, and runs Heretic on Qwen3-4B with Qwen thinking mode disabled. Disabling thinking avoids generating `<think>...</think>` blocks during Heretic's repeated generations, which should save wall time.

Use a GPU runtime: **Runtime -> Change runtime type -> GPU**. For long runs, mount Google Drive so Optuna checkpoints survive runtime resets.

In [ ]:
# @title Runtime knobs
REPO_URL = "https://github.com/rowan-dauria/heretic.git"  # @param {type:"string"}
REPO_BRANCH = "codex/heretic-touched-layers"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-4B"  # @param {type:"string"}

# Use bnb_4bit for Colab T4/L4 safety. On A100/H100, "none" is faster if VRAM is ample.
QUANTIZATION = "bnb_4bit"  # @param ["bnb_4bit", "none"]

# Start smaller for a smoke run, then increase once the setup is proven.
N_TRIALS = 40  # @param {type:"integer"}
N_STARTUP_TRIALS = 12  # @param {type:"integer"}
MAX_RESPONSE_LENGTH = 80  # @param {type:"integer"}
MAX_BATCH_SIZE = 32  # @param {type:"integer"}
BATCH_SIZE = 0  # @param {type:"integer"}

# Smaller prompt slices make quick test runs much cheaper. Increase for final runs.
TRAIN_SLICE = "train[:200]"  # @param {type:"string"}
EVAL_SLICE = "test[:80]"  # @param {type:"string"}

USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_RUN_DIR = "/content/drive/MyDrive/heretic-qwen3-4b"  # @param {type:"string"}
LOCAL_RUN_DIR = "/content/heretic-qwen3-4b"  # @param {type:"string"}

ENABLE_THINKING = False

## Check the GPU

If this does not show a CUDA GPU, switch the runtime before continuing.

In [ ]:
!nvidia-smi

## Clone and install the fork

The notebook installs the local checkout in editable mode so it uses your fork rather than the PyPI release.

In [ ]:
from pathlib import Path
import os
import shutil

repo_dir = Path("/content/heretic")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone --branch "$REPO_BRANCH" "$REPO_URL" "$repo_dir" || (git clone "$REPO_URL" "$repo_dir" && cd "$repo_dir" && git checkout "$REPO_BRANCH")
%cd /content/heretic
!python -m pip install -q --upgrade pip
!python -m pip install -q -e .

## Optional Hugging Face login

`Qwen/Qwen3-4B` and the default datasets are public, so this is usually unnecessary. Log in if you switch to a gated/private model, want to upload results, or hit Hub rate limits.

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login

    token = userdata.get("HF_TOKEN")
    if token:
        login(token=token)
        print("Logged in with HF_TOKEN from Colab secrets.")
    else:
        print("No HF_TOKEN Colab secret found; continuing anonymously.")
except Exception as exc:
    print(f"Skipping Hugging Face login: {exc}")

## Prepare the run directory and config

The important line is `enable_thinking = false`. Heretic's patched fork passes this into `tokenizer.apply_chat_template(...)`, so Qwen3 should not spend tokens on hidden reasoning blocks.

In [ ]:
from pathlib import Path
import textwrap

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    run_dir = Path(DRIVE_RUN_DIR)
else:
    run_dir = Path(LOCAL_RUN_DIR)

run_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir = run_dir / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

config = f"""
model = "{MODEL_ID}"
dtypes = ["bfloat16", "float16", "auto"]
quantization = "{QUANTIZATION}"
device_map = "auto"
offload_outputs_to_cpu = true

batch_size = {BATCH_SIZE}
max_batch_size = {MAX_BATCH_SIZE}
max_response_length = {MAX_RESPONSE_LENGTH}

enable_thinking = {str(ENABLE_THINKING).lower()}
response_prefix = ""

n_trials = {N_TRIALS}
n_startup_trials = {N_STARTUP_TRIALS}
study_checkpoint_dir = "{checkpoint_dir}"

# Save LoRA adapters by default if you choose to export after optimization.
export_strategy = "adapter"

[good_prompts]
dataset = "mlabonne/harmless_alpaca"
split = "{TRAIN_SLICE}"
column = "text"

[bad_prompts]
dataset = "mlabonne/harmful_behaviors"
split = "{TRAIN_SLICE}"
column = "text"

[good_evaluation_prompts]
dataset = "mlabonne/harmless_alpaca"
split = "{EVAL_SLICE}"
column = "text"

[bad_evaluation_prompts]
dataset = "mlabonne/harmful_behaviors"
split = "{EVAL_SLICE}"
column = "text"
"""

(run_dir / "config.toml").write_text(textwrap.dedent(config).strip() + "\n")
print(f"Run directory: {run_dir}")
print((run_dir / "config.toml").read_text())

## Sanity-check the chat template

This prints the rendered prompt tail. With thinking disabled, Qwen should not include an opening `<think>` instruction in the generated assistant prefix.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
rendered = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Give me a one-sentence greeting."}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)
print(rendered[-600:])

## Run Heretic

This cell is interactive after optimization: select a trial, then choose whether to save an adapter, chat, benchmark, or exit. If the run is interrupted, rerun the notebook with the same `run_dir`; Heretic will offer to resume from the checkpoint.

In [ ]:
import os
os.chdir(run_dir)
!pwd
!PYTORCH_ALLOC_CONF=expandable_segments:True heretic

## Export notes

With `export_strategy = "adapter"`, choose **Save the model to a local folder** after selecting a trial and provide a path inside `run_dir`, for example `/content/heretic-qwen3-4b/qwen3-4b-heretic-adapter`. Adapter export is much lighter than merging, especially if the run used 4-bit quantization.